
# Label Variations JSON Generator

**task:** Provide a `{label: weight}` mapping, and get back a JSON response from an LLM containing each base label and a set of slight variations. Higher-weight labels receive more variations.

- **Backends supported:** OpenAI(api) or Ollama(local).
- **Output:** The model is instructed to return JSON that conforms to a simple schema.

> **Setup:**
> - OpenAI: `pip install --upgrade "openai>=1.0.0"` and set `OPENAI_API_KEY`
> - Ollama: install and run Ollama, e.g. `ollama pull llama3.1`; set `USE_OLLAMA=1`


In [4]:

# --- USER INPUTS ---

# Base labels and weights (example)
LABEL_WEIGHTS = {
    "happy": 0.6,
    "sad": 0.1,
    "whimsical": 0.2,
    "mysterious": 0.1,
}

# Total number of variations across ALL labels (not counting the base labels)
TOTAL_VARIATIONS = 20

# Allow multi-word phrases? e.g., "light-hearted", "bittersweet wonder"
ALLOW_MULTIWORD = False

# LLM Settings
TEMPERATURE = 0.8

# Backend selection
# - If using OpenAI, ensure OPENAI_API_KEY is in env.
# - If using Ollama, set USE_OLLAMA=1 in env (and OLLAMA_MODEL if desired).
OPENAI_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = "llama3.1"  


In [ ]:

import os, math, json
from typing import Dict

USE_OLLAMA = bool(os.environ.get("USE_OLLAMA", "0") not in ["0","false","False",""])

def proportional_allocation(label_weights: Dict[str, float], total: int) -> Dict[str, int]:
    import math
    items = [(lab, max(0.0, float(w))) for lab, w in label_weights.items()]
    total_w = sum(w for _, w in items)
    if total_w == 0:
        base = total // max(1, len(items))
        counts = {lab: base for lab, _ in items}
        remainder = total - base * len(items)
        labs = list(counts.keys())
        for i in range(remainder):
            counts[labs[i % len(labs)]] += 1
        return counts
    ideals = [(lab, (w / total_w) * total) for lab, w in items]
    floor_counts = {lab: int(math.floor(frac)) for lab, frac in ideals}
    used = sum(floor_counts.values())
    remainder = total - used
    fracs = sorted([(lab, frac - math.floor(frac)) for lab, frac in ideals],
                   key=lambda x: x[1], reverse=True)
    for i in range(remainder):
        floor_counts[fracs[i % len(fracs)][0]] += 1
    return floor_counts

def build_json_only_prompt(label_weights: Dict[str, float],
                           per_label_counts: Dict[str, int],
                           allow_multiword: bool,
                           temperature: float) -> str:
    allow_str = "may be multi-word" if allow_multiword else "must be single-word only"
    lines = []
    lines.append("You are a controlled JSON generator.")
    lines.append("Return ONLY valid JSON (UTF-8), with no leading/trailing text, no comments, and no markdown. Do not include triple backticks.")
    lines.append('Schema: {"labels":[{"base": "<string>", "weight": <number>, "variations": ["<string>", ...]}]}')
    lines.append("Requirements:")
    lines.append(f"- For each base label, produce EXACTLY the number of variations specified below.")
    lines.append(f"- Variations should be slight semantic neighbors (synonyms/near-synonyms, tonal or stylistic variants). Variations {allow_str}.")
    lines.append("- Avoid rare, archaic, or misspelled words. Use common, contemporary vocabulary.")
    lines.append("- No duplicates across variations of the same base.")
    lines.append("- Variations should reflect the relative weight distribution: higher-weight labels got more slots; fill them with high-quality, diverse terms.")
    lines.append("Input labels (with required variation counts):")
    for lab, cnt in per_label_counts.items():
        w = float(label_weights.get(lab, 0.0))
        lines.append(f'- base="{lab}", weight={w}, required_variations={cnt}')
    lines.append("Output MUST be JSON and match the schema exactly.")
    return "\\n".join(lines)

def call_openai_json(prompt: str, model: str, temperature: float) -> str:
    try:
        from openai import OpenAI
    except Exception as e:
        raise RuntimeError("OpenAI v1 SDK not available. Install with: pip install --upgrade 'openai>=1.0.0'") from e
    client = OpenAI()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}],
        temperature=temperature,
        max_tokens=800,
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

def call_ollama_json(prompt: str, model: str, temperature: float) -> str:
    try:
        import requests
    except Exception as e:
        raise RuntimeError("`requests` not installed. pip install requests") from e
    url = "http://localhost:11434/api/generate"
    data = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    }
    r = requests.post(url, json=data, timeout=600)
    r.raise_for_status()
    return r.json().get("response","")

def generate_label_variations_json(label_weights: Dict[str, float],
                                   total_variations: int,
                                   allow_multiword: bool,
                                   temperature: float,
                                   openai_model: str,
                                   ollama_model: str) -> str:
    counts = proportional_allocation(label_weights, total_variations)
    prompt = build_json_only_prompt(label_weights, counts, allow_multiword, temperature)

    if USE_OLLAMA:
        raw = call_ollama_json(prompt, ollama_model, temperature)
    else:
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY not set. Or set USE_OLLAMA=1 to use a local model.")
        raw = call_openai_json(prompt, openai_model, temperature)

    return raw


In [6]:

# === RUN ===
try:
    raw_json = generate_label_variations_json(
        label_weights=LABEL_WEIGHTS,
        total_variations=TOTAL_VARIATIONS,
        allow_multiword=ALLOW_MULTIWORD,
        temperature=TEMPERATURE,
        openai_model=OPENAI_MODEL,
        ollama_model=OLLAMA_MODEL,
    )
    print(raw_json)
except Exception as e:
    print("Generation error:", e)
    print("\nTip: Ensure OpenAI v1 SDK is installed and OPENAI_API_KEY is set, or set USE_OLLAMA=1 with a running Ollama model.")


{
  "labels": [
    {
      "base": "happy",
      "weight": 0.6,
      "variations": [
        "joyful",
        "cheerful",
        "content",
        "delighted",
        "pleased",
        "elated",
        "blissful",
        "glad",
        "jovial",
        "upbeat",
        "radiant",
        "satisfied"
      ]
    },
    {
      "base": "sad",
      "weight": 0.1,
      "variations": [
        "unhappy",
        "sorrowful"
      ]
    },
    {
      "base": "whimsical",
      "weight": 0.2,
      "variations": [
        "playful",
        "fanciful",
        "quirky",
        "capricious"
      ]
    },
    {
      "base": "mysterious",
      "weight": 0.1,
      "variations": [
        "enigmatic",
        "cryptic"
      ]
    }
  ]
}
